# Pydantic AI Chatbot API - Interactive Notebook Client

This notebook demonstrates and tests all features of the permission-aware chatbot system:
1. **In-Memory Chat History** across multi-turn sessions.
2. **Button Trigger Human-in-the-Loop (HITL)**: Tool pauses execution and sends format buttons (`["Executive Summary", "Full Raw Logs", "CSV Format"]`) to the UI. The user clicks a button, and the choice is **injected in the background without the LLM seeing or guessing the format parameter**.
3. **Out-of-Band Report Delivery**: The generated report data is sent directly to the client via `ConfidentialDelivery`, keeping raw data completely out of the LLM context.
4. **Verbal Human-in-the-Loop (HITL)**: Tool instructs the model to verbally ask for user confirmation in natural chat before executing critical actions.
5. **Confidential Out-of-Band Delivery**: LLM only receives a redacted receipt, while the user's client receives the full unredacted payload.
6. **Dynamic Tool & Topic Filtering**: Tools and topics the user lacks permission for are filtered out before the LLM can see them.
7. **Message History & Tool Inspection (`.all_messages()`)**: Inspect which tools the model called, arguments passed, thinking/reasoning process, and tool outputs.

## 1. Setup & Connection
Make sure your FastAPI server is running in a terminal:
```bash
uvicorn main:app --reload --port 8000
```

In [ ]:
from client import ChatClient
import json

# Connect client as Alice (Senior Analyst / Admin)
client = ChatClient(base_url="http://127.0.0.1:8000", user_id="user_alice")

# Inspect available mock users and their clearances
print("Available Mock Users and Clearances:")
print(json.dumps(client.get_users(), indent=2))

## 2. Test Multi-Turn Chat History
The client maintains `session_id`. Notice how the bot remembers context from turn 1 in turn 2.

In [ ]:
# Turn 1: Share a piece of information
client.chat("Hello! My favorite project code name is 'Project Phoenix'. Please remember this.")

In [ ]:
# Turn 2: Verify memory retention across turns
client.chat("What is my favorite project code name?")

## 3. Test Button Trigger Human-In-The-Loop & Out-of-Band Report Export
**Key architectural note**: The LLM's schema for `request_report_export` only accepts `(report_name: str)`. It has **NO** `format` parameter!
1. The model calls `request_report_export(report_name="financial_q3")`.
2. The tool pauses and sends button options: `["Executive Summary", "Full Raw Logs", "CSV Format"]`.
3. You click a button (enter 1, 2, or 3 below).
4. Your choice is injected **in the background** into user context via `/chat/resume`.
5. The tool generates the report and delivers it **out-of-band via ConfidentialDelivery** directly to your secure display!
6. The LLM receives only a confirmation receipt—neither the format choice nor the raw table data ever enters the model's context window!

In [ ]:
# Triggers format button prompt in notebook
client.chat("Please export the financial_q3 report for me.")

## 4. Test Verbal Human-In-The-Loop (Critical System Action)
Unlike the button tool, this is **pure conversational confirmation**:
1. The `execute_critical_system_action` tool sees `confirmed=False`.
2. The LLM verbally asks you in chat: *"Are you sure you want to restart the primary cluster?"*
3. In the next turn, you reply *"Yes, proceed"*.
4. The LLM inspects chat history, sees your verbal consent, and calls the tool with `confirmed=True`!

In [ ]:
# Step 1: Request critical action (LLM will ask you for confirmation in natural language)
client.chat("Restart the primary cluster.")

In [ ]:
# Step 2: Give verbal confirmation in conversation
client.chat("Yes, I confirm. Please proceed with restarting the primary cluster.")

## 5. Test Confidential Out-of-Band Delivery & LLM Redaction
The tool retrieves sensitive data. Notice:
- The LLM only receives a **redacted receipt**.
- The secret key material is delivered to your client's secure payload box without the LLM ever seeing the tokens.

In [ ]:
# Alice is authorized for quantum_keys
client.chat("Fetch the confidential documentation for quantum_keys.")

## 6. Inspect Conversation History & Model Tool Calls (`.all_messages()`)
Inspect the exact tools the model executed, tool arguments, tool outputs, and reasoning tokens across the session.

In [ ]:
# Inspect full history for Alice's current session
client.print_history()

## 7. Test Permission Filtering with Bob (Junior Analyst)
Switch users to **Bob**. Bob:
- Does NOT have `execute_critical_system_action` or `database_query` (LLM cannot see these tools).
- Does NOT have access to `quantum_keys` (Confidential RAG denies access).
- DOES have access to `payroll_audit` and report export.

In [ ]:
client.set_user("user_bob")

# 1. Bob tries critical system action (Tool is filtered out, LLM doesn't have it)
client.chat("Restart the primary cluster.")

In [ ]:
# 2. Bob tries unauthorized confidential topic (In-tool denial)
client.chat("Fetch the confidential documentation for quantum_keys.")

In [ ]:
# 3. Bob accesses authorized confidential topic (Success)
client.chat("Fetch the confidential documentation for payroll_audit.")

## 8. Interactive Free-form Chat Loop
Run the cell below to chat freely with the bot inside your notebook. Type `history` to inspect messages, or `exit` to stop.

In [ ]:
# Switch back to Alice or keep Bob, and start interactive chat:
# client.set_user("user_alice")
# client.interactive_loop()